
# Reservoir DDMS – REST & ETP demo

This unified notebook keeps your original variable model and adds robust ETP session handling:
- Waits for the ETP handshake to complete before discovery
- Retries the first `getResources` call if the server is still warming up
- Uses the selected `dataspace_path` directly for the EML dataspace URI


In [9]:
from __future__ import annotations
import os, time, json, base64, logging, urllib.parse as _url
from pathlib import Path
from typing import Dict, Any
import requests

logging.basicConfig(
    level=getattr(logging, os.environ.get("LOG_LEVEL", "INFO").upper(), logging.INFO),
    format="%(asctime)s.%(msecs)03d %(levelname)s %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("rddms")
INFO = lambda m: log.info(m)
ERROR = lambda m: log.error(m)

# Environment & defaults (host only for base URL)
OSDU_BASE_URL     = os.environ.get("OSDU_BASE_URL", "equinordev.energy.azure.com")
DATA_PARTITION_ID = os.environ.get("DATA_PARTITION_ID", "data")
AZURE_TENANT_ID   = os.environ.get("AZURE_TENANT_ID", "3aa4a235-b6e2-48d5-9195-7fcf05b459b0")
AZURE_CLIENT_ID   = os.environ.get("AZURE_CLIENT_ID", "ebd2bfee-ecba-47b7-a33c-017d0131879d")
AZURE_SCOPE       = os.environ.get("AZURE_SCOPE", "7daee810-3f78-40c4-84c2-7a199428de18/.default openid offline_access")
DATASPACE         = os.environ.get("DATASPACE", "maap/drogon")

# Derived endpoints
OSDU_BASE      = f"https://{OSDU_BASE_URL}"
RDDMS_REST_API = f"{OSDU_BASE}/api/reservoir-ddms/v2"
RDDMS_ETP_API  = f"wss://{OSDU_BASE_URL}/api/reservoir-ddms-etp/v2/"  # keep trailing slash

# Services
entitlements_endpoint = f"{OSDU_BASE}/api/entitlements/v2"
legal_endpoint        = f"{OSDU_BASE}/api/legal/v1"
search_endpoint       = f"{OSDU_BASE}/api/search/v2"

# Entitlements / Legal defaults
entitlements_domain = f"{DATA_PARTITION_ID}.dataservices.energy"
DEFAULT_LEGAL_TAG   = f"{DATA_PARTITION_ID}-equinor-private-default"
DEFAULT_OTHER_RELEVANT_DATA_COUNTRIES = ["NO"]
DEFAULT_OWNERS  = [f"data.default.owners@{entitlements_domain}"]
DEFAULT_VIEWERS = [f"data.default.viewers@{entitlements_domain}"]

# Dataspace path & encoded
dataspace_path = DATASPACE
dataspace_name = _url.quote(dataspace_path, safe="")

# Token plumbing (robust: env first, then .env fallback; search parents for .env)
_TOKEN_URL = f"https://login.microsoftonline.com/{AZURE_TENANT_ID}/oauth2/v2.0/token" if AZURE_TENANT_ID else None
_TOKEN = {"access": None, "exp": 0.0}

def _find_dotenv(start: str | Path = ".") -> Path | None:
    p = Path(start).resolve()
    for candidate in [p] + list(p.parents):
        f = candidate / ".env"
        if f.is_file():
            INFO(f"Found .env at: {f}")
            return f
    INFO(".env not found in parents")
    return None

def _load_dotenv_file(path: str | Path) -> Dict[str,str]:
    data: Dict[str,str] = {}
    try:
        with open(path, "r", encoding="utf-8") as fh:
            for line in fh:
                line = line.strip()
                if not line or line.startswith("#"):
                    continue
                if "=" not in line:
                    continue
                k, v = line.split("=", 1)
                k = k.strip()
                v = v.strip().strip('"').strip("'")
                data[k] = v
    except Exception as e:
        INFO(f"Failed to read .env: {e}")
    return data

_dotenv_cache: Dict[str,str] | None = None
def _get_env_or_dotenv(key: str):
    # 1) try environment with casings
    for k in (key, key.upper(), key.lower()):
        v = os.environ.get(k)
        if v:
            INFO(f"Using env var: {k}")
            return v
    # 2) try nearest .env (search upward)
    global _dotenv_cache
    if _dotenv_cache is None:
        f = _find_dotenv(os.getcwd())
        _dotenv_cache = _load_dotenv_file(f) if f else {}
    for k in (key, key.upper(), key.lower()):
        v = _dotenv_cache.get(k)
        if v:
            INFO(f"Using .env value for: {k}")
            return v
    return None

def _require_env(n: str) -> str:
    v = _get_env_or_dotenv(n)
    if not v:
        raise RuntimeError(f"Environment variable '{n}' is required (looked in environment and nearest .env).")
    return v

def _decode_jwt(tok: str) -> Dict[str,Any]:
    try:
        p = tok.split(".")[1]; p = p + "=="[:(4 - len(p) % 4) % 4]
        return json.loads(base64.urlsafe_b64decode(p).decode())
    except Exception:
        return {}

def get_access_token() -> str:
    if _TOKEN["access"] and _TOKEN["exp"] > time.time() + 60:
        return _TOKEN["access"]
    if not _TOKEN_URL:
        raise RuntimeError("AZURE_TENANT_ID must be set")
    refresh = _get_env_or_dotenv("REFRESH_TOKEN") or _get_env_or_dotenv("refresh_token")
    if not refresh:
        raise RuntimeError("REFRESH_TOKEN not found in environment or any parent .env (set REFRESH_TOKEN or refresh_token)")
    data = {
        "grant_type": "refresh_token",
        "client_id": AZURE_CLIENT_ID,
        "scope": AZURE_SCOPE,
        "refresh_token": refresh,
    }
    INFO("Requesting AAD access_token via refresh_token…")
    r = requests.post(_TOKEN_URL, data=data, headers={"Content-Type": "application/x-www-form-urlencoded"}, timeout=90)
    r.raise_for_status()
    js = r.json(); tok = js.get("access_token")
    if not tok:
        raise RuntimeError(js.get("error_description") or js.get("error") or "No access_token")
    _TOKEN["access"] = tok; _TOKEN["exp"] = time.time() + float(js.get("expires_in", 3600)) - 60.0
    os.environ["access_token"] = tok; INFO("Access token acquired.")
    return tok

# alias kept for compatibility with later cells
get_token = get_access_token

def bearer() -> str:
    return f"Bearer {get_access_token()}"

def whoami() -> Dict[str,str]:
    c = _decode_jwt(get_access_token())
    ident = str(c.get("preferred_username") or c.get("upn") or c.get("email") or c.get("oid") or "user")
    base = "".join(ch.lower() if ch.isalnum() else "-" for ch in (ident.split("@")[0] if "@" in ident else ident)).strip("-") or "user"
    import hashlib as _h
    h = _h.sha256(ident.encode()).hexdigest()[:8]
    return {"display": ident, "email": ident if "@" in ident else "", "handle": f"{base}-{h}"}

def build_headers() -> Dict[str,str]:
    return {"Authorization": bearer(), "data-partition-id": DATA_PARTITION_ID, "content-type": "application/json", "accept": "application/json"}

def get_rddms_header() -> Dict[str,str]:
    return build_headers()

# Quick smoke to Search (safe; will log failures)
try:
    smoke = requests.post(f"{search_endpoint}/query", headers=build_headers(), json={"kind":"*:*:*:*","query":"*","limit":1}, timeout=60)
    smoke.raise_for_status(); INFO("AUTH SUCCESS — Search reachable.")
except Exception as e:
    ERROR(f"AUTH FAILED: {e}")

user = whoami(); INFO(f"Connected as: {user['display']} (handle={user['handle']})")
INFO(f"RDDMS REST base: {RDDMS_REST_API}")
INFO(f"RDDMS ETP base: {RDDMS_ETP_API}")
INFO(f"dataspace path: {dataspace_path} \\n encoded: {dataspace_name}")

07:39:12.360 INFO Using env var: REFRESH_TOKEN
07:39:12.361 INFO Requesting AAD access_token via refresh_token…
07:39:12.363 DEBUG Starting new HTTPS connection (1): login.microsoftonline.com:443
07:39:12.645 DEBUG https://login.microsoftonline.com:443 "POST /3aa4a235-b6e2-48d5-9195-7fcf05b459b0/oauth2/v2.0/token HTTP/1.1" 200 4869
07:39:12.646 INFO Access token acquired.
07:39:12.649 DEBUG Starting new HTTPS connection (1): equinorswedev.energy.azure.com:443
07:39:12.989 DEBUG https://equinorswedev.energy.azure.com:443 "POST /api/search/v2/query HTTP/1.1" 200 None
07:39:12.991 INFO AUTH SUCCESS — Search reachable.
07:39:12.992 INFO Connected as: MAAP@equinor.com (handle=maap-6eb37a71)
07:39:12.992 INFO RDDMS REST base: https://equinorswedev.energy.azure.com/api/reservoir-ddms/v2
07:39:12.993 INFO RDDMS ETP base: wss://equinorswedev.energy.azure.com/api/reservoir-ddms-etp/v2/
07:39:12.993 INFO dataspace path: maap/drogon \n encoded: maap%2Fdrogon


In [10]:

# --- Rich API logging helper (redacts token; truncates JSON; summarizes arrays) ---
import time as _t
import json as _json
import requests

def _redact_auth(hdrs: dict) -> dict:
    safe = {}
    for k, v in (hdrs or {}).items():
        if isinstance(k, str) and k.lower() == 'authorization' and isinstance(v, str):
            parts = v.split(); tok = parts[-1] if parts else ''
            safe[k] = f"Bearer <{len(tok)} bytes>"
        else:
            safe[k] = v
    return safe

def api_request(method: str, url: str, *, headers=None, params=None, json_body=None,
                data=None, timeout=120, max_json_bytes=4096, tag=None):
    hdr = headers or get_rddms_header()
    safe_hdr = _redact_auth(hdr)

    INFO(f"[{method}] {url} tag={tag or ''}")
    INFO(f" headers={safe_hdr}")
    if params: INFO(f" params={params}")
    if json_body is not None:
        try:
            body_txt = _json.dumps(json_body)
        except Exception:
            body_txt = str(json_body)
        if len(body_txt) > 1000:
            body_txt = body_txt[:1000] + f" … (+{len(body_txt)-1000} chars)"
        INFO(f" json={body_txt}")
    elif data is not None:
        body_len = len(data) if isinstance(data, (bytes, bytearray)) else len(str(data))
        INFO(f" data=<{body_len} chars>")

    t0 = _t.perf_counter()
    resp = requests.request(method, url, headers=hdr, params=params, json=json_body, data=data, timeout=timeout)
    dt = (_t.perf_counter() - t0) * 1000.0

    ctype = resp.headers.get('Content-Type', '')
    INFO(f"→ status={resp.status_code} content-type='{ctype}' bytes={len(resp.content or b'')} time={dt:.1f}ms")

    js = None
    try:
        js = resp.json()
    except Exception:
        pass

    if isinstance(js, dict):
        trace = js.get('traceId') or js.get('traceid') or js.get('trace_id')
        if trace:
            INFO(f"→ traceId={trace}")

    if js is not None:
        if '/arrays' in url and isinstance(js, dict):
            try:
                dims = js['data']['dimensions']
                n = len(js['data']['data'])
                INFO(f"→ arrays meta: dimensions={dims} length={n}")
            except Exception:
                txt = _json.dumps(js, indent=2)
                extra = len(txt) - max_json_bytes
                preview = txt[:max_json_bytes] + (f"… (+{extra} bytes more)" if extra>0 else '')
                INFO(f"→ json body (truncated):{preview}")
        else:
            txt = _json.dumps(js, indent=2)
            extra = len(txt) - max_json_bytes
            preview = txt[:max_json_bytes] + (f"… (+{extra} bytes more)" if extra>0 else '')
            INFO(f"→ json body (truncated):{preview}")
    return resp


In [12]:

# --- Utilities ---
import pandas as pd
import numpy as np
from IPython.display import display, HTML

def pretty_print_panda_frame(df, title=None, transpose=False, show_index=False, show_header=True):
    styled = (df.transpose() if transpose else df).style.set_properties(**{'text-align': 'left'})
    if title: display(HTML(f"<h3 style='margin-bottom: 0px'>{title}</h3>"))
    if not show_index: styled = styled.hide(axis='index')
    if not show_header: styled = styled.set_table_styles([{'selector': 'thead', 'props': [('display', 'none')]}])
    else: styled = styled.set_table_styles([{'selector': 'th', 'props': [('text-align', 'left')]}])
    display(styled)


ModuleNotFoundError: No module named 'pandas'

### 1. Locate RDDMS (authenticated)

In [ ]:

root = api_request('GET', f"{RDDMS_REST_API}/", tag='rddms-root')
try:
    root.raise_for_status(); print("RDDMS REST API is responding.")
except Exception as e:
    print("Error connecting to RDDMS REST API.")
    print(e)
    if getattr(e, 'response', None) is not None:
        print(f"Response Body: {e.response.text}")
print(f"RDDMS REST API at: {RDDMS_REST_API}/")
print(f"RDDMS ETP API at: {RDDMS_ETP_API}")


### 2. List available dataspaces

In [ ]:

all_ds = api_request('GET', f"{RDDMS_REST_API}/dataspaces", tag='list-dataspaces')
all_ds.raise_for_status()
rows = [{"Path": x.get("path"), "URI": x.get("uri"), "Created": x.get("storeCreated"), "Updated": x.get("storeLastWrite"),
         "Locked": (x.get("customData") or {}).get("locked")} for x in all_ds.json()]
pretty_print_panda_frame(pd.DataFrame(rows), title=f"Dataspaces ({len(rows)})")


### 3. Select dataspace `DATASPACE`

In [ ]:

import urllib.parse
try:
    SelectedDataspace = next(item for item in all_ds.json() if item.get("path") == dataspace_path)
    dataspace_name = urllib.parse.quote(SelectedDataspace['path'], safe="")
    print(f"DATASPACE '{SelectedDataspace['path']}' SELECTED")
except StopIteration:
    raise SystemExit(f"DATASPACE '{dataspace_path}' NOT FOUND")


### 4. List dataspace resources

In [ ]:

res = api_request('GET', f"{RDDMS_REST_API}/dataspaces/{dataspace_name}/resources", tag='list-resources')
res.raise_for_status()
flat = pd.json_normalize(res.json()).to_dict(orient='records')
pretty_print_panda_frame(pd.DataFrame.from_dict(flat), title="Resources (types)")


### 5. List horizons (`resqml20.obj_Grid2dRepresentation`)

In [ ]:

resqml_datatype = 'resqml20.obj_Grid2dRepresentation'
hr = api_request('GET', f"{RDDMS_REST_API}/dataspaces/{dataspace_name}/resources/{resqml_datatype}", tag='list-horizons')
hr.raise_for_status()
hlist = hr.json() or []
if not hlist:
    print("No horizons found in this dataspace.")
else:
    tbl = [{"Name": x.get("name"), "URI": x.get("uri"), "Created": (x.get("customData") or {}).get("created"),
            "Creator": (x.get("customData") or {}).get("creator") } for x in hlist]
    pretty_print_panda_frame(pd.DataFrame(tbl), title=f"Horizons ({len(tbl)})")


### 6. Read first horizon: metadata, geometry, arrays

In [ ]:

import urllib.parse

HAS_HORIZON = len(hr.json() or []) > 0
horizon_uuid = None
if HAS_HORIZON:
    first_uri = hr.json()[0]['uri']
    horizon_uuid = urllib.parse.quote(first_uri.split('(')[-1].replace(')',''))
    print('Selected Horizon uuid:', horizon_uuid)
else:
    print('Skipping: no horizons available.')

# ---- Geometry helpers (variant‑tolerant) ----

def _get(d: dict, path: list[str]):
    cur = d
    for k in path:
        if not isinstance(cur, dict):
            return None
        cur = cur.get(k)
    return cur

def _parse_content_type_to_type(content_type: str) -> str | None:
    if not isinstance(content_type, str):
        return None
    low = content_type.lower()
    if 'type=' not in low:
        return None
    return content_type.split('type=', 1)[-1].strip()

def _resolve_lattice_from_points_obj(points_obj: dict) -> dict | None:
    if not isinstance(points_obj, dict):
        return None
    ptype = points_obj.get('$type') or points_obj.get('type')
    # Case 0: ZValueArray where SupportingGeometry IS the lattice
    if ptype == 'resqml20.Point3dZValueArray':
        sg = points_obj.get('SupportingGeometry')
        if isinstance(sg, dict) and sg.get('$type') == 'resqml20.Point3dLatticeArray':
            return sg
    # Case 1: already lattice
    if ptype == 'resqml20.Point3dLatticeArray':
        return points_obj
    # Case 2: other inline patterns
    if ptype == 'resqml20.Point3dZValueArray':
        for p in (
            ['SupportingGeometry','Points'],
            ['SupportingGeometry','Grid2dPatch','Geometry','Points'],
            ['SupportingGeometry','_data','Grid2dPatch','Geometry','Points'],
            ['SupportingGeometry','SupportingRepresentation','_data','Grid2dPatch','Geometry','Points'],
            ['SupportingRepresentation','_data','Grid2dPatch','Geometry','Points'],
        ):
            cand = _get(points_obj, p)
            if isinstance(cand, dict) and cand.get('$type') == 'resqml20.Point3dLatticeArray':
                return cand
    return None

def _fetch_supporting_representation(points_obj: dict, *, dataspace_name: str, resqml_datatype: str) -> dict | None:
    srep = points_obj.get('SupportingRepresentation') or _get(points_obj, ['SupportingGeometry','SupportingRepresentation'])
    if not isinstance(srep, dict):
        return None
    uuid = srep.get('UUID') or srep.get('Uuid') or srep.get('uuid')
    if not uuid:
        return None
    ct = srep.get('ContentType') or srep.get('contentType') or ''
    typ = _parse_content_type_to_type(ct) or resqml_datatype or 'resqml20.obj_Grid2dRepresentation'
    params = {'$format':'json','referencedContent':'true','arrayMetadata':'false','arrayValues':'false'}
    url = f"{RDDMS_REST_API}/dataspaces/{dataspace_name}/resources/{typ}/{uuid}"
    r = api_request('GET', url, params=params, headers=get_rddms_header(), tag='supporting-repr')
    if r.status_code != 200:
        INFO(f"supporting-repr fetch failed: status={r.status_code}")
        return None
    try:
        js = r.json(); grid_patch = js[0]['Grid2dPatch']; pts = grid_patch['Geometry']['Points']
    except Exception as e:
        INFO(f"supporting-repr parse error: {e}")
        return None
    return _resolve_lattice_from_points_obj(pts)

def _vectors_from_lattice(lattice: dict, grid_patch: dict):
    origin = [lattice['Origin']['Coordinate1'], lattice['Origin']['Coordinate2']]
    offsets = lattice.get('Offset', [])
    spacing = [ (o.get('Spacing') or {}).get('Value') for o in offsets ]
    vectors = [ [o['Offset']['Coordinate1'], o['Offset']['Coordinate2']] for o in offsets ]
    sizes = [grid_patch['FastestAxisCount'], grid_patch['SlowestAxisCount']]
    if len(spacing)==2 and len(vectors)==2 and all(s is not None for s in spacing):
        u_vec = [vectors[0][0]*spacing[0], vectors[0][1]*spacing[0]]
        v_vec = [vectors[1][0]*spacing[1], vectors[1][1]*spacing[1]]
    else:
        if len(vectors)<2:
            vectors = vectors + [[1.0,0.0],[0.0,1.0]]
        u_vec, v_vec = vectors[0], vectors[1]
    return u_vec, v_vec, sizes, origin

def extract_grid_geometry(json_data, *, dataspace_name: str, resqml_datatype: str, allow_index_fallback: bool=True):
    try:
        grid_patch = json_data[0]['Grid2dPatch']
        geom = grid_patch['Geometry']
        points = geom['Points']
        ptype = points.get('$type')
        lattice = _resolve_lattice_from_points_obj(points)
        source = 'inline-lattice'
        if lattice is None and ptype == 'resqml20.Point3dZValueArray':
            lattice = _fetch_supporting_representation(points, dataspace_name=dataspace_name, resqml_datatype=resqml_datatype)
            source = 'supporting-fetch'
        if lattice is not None:
            u_vec, v_vec, sizes, origin = _vectors_from_lattice(lattice, grid_patch)
            import numpy as _np
            ortho = _np.isclose(_np.dot(_np.array(u_vec), _np.array(v_vec)), 0.0, atol=1e-6)
            INFO(f"Grid2d vectors orthogonal? {'Yes' if ortho else 'No'}")
            return {"Origin": origin, "Vector": [u_vec, v_vec], "Size": sizes, "Source": source}
        if allow_index_fallback:
            sizes = [grid_patch['FastestAxisCount'], grid_patch['SlowestAxisCount']]
            INFO("XY lattice unresolved — using index grid fallback (i,j).")
            return {"Origin":[0.0,0.0], "Vector":[[1.0,0.0],[0.0,1.0]], "Size": sizes, "Source": "index-fallback"}
        raise KeyError(f"Unsupported or unresolved Points layout: $type={ptype}")
    except (KeyError, IndexError, TypeError) as e:
        raise RuntimeError(f"extract_grid_geometry failed: {e}")


def process_horizon_metadata(RDDMS_REST_API, dataspace_name, resqml_datatype, horizon_uuid, headers):
    params = {'$format': 'json', 'arrayMetadata': 'false', 'arrayValues': 'false', 'referencedContent': 'true'}
    r = api_request('GET', f"{RDDMS_REST_API}/dataspaces/{dataspace_name}/resources/{resqml_datatype}/{horizon_uuid}", params=params, headers=headers, tag='horizon-metadata')
    r.raise_for_status()
    geometry = extract_grid_geometry(r.json(), dataspace_name=dataspace_name, resqml_datatype=resqml_datatype)
    arrays = api_request('GET', f"{RDDMS_REST_API}/dataspaces/{dataspace_name}/resources/{resqml_datatype}/{horizon_uuid}/arrays", params=params, headers=headers, tag='horizon-arrays')
    arrays.raise_for_status()
    return geometry, arrays

horizon_geometry, arrays_response = (None, None)
if HAS_HORIZON:
    horizon_geometry, arrays_response = process_horizon_metadata(RDDMS_REST_API, dataspace_name, resqml_datatype, horizon_uuid, headers=get_rddms_header())
    geom_df = pd.DataFrame(horizon_geometry)
    pretty_print_panda_frame(geom_df, title="Horizon Geometry", transpose=True, show_index=True, show_header=False)


### 7. Read depth array & stats

In [ ]:

if HAS_HORIZON and arrays_response is not None:
    import urllib.parse
    def process_horizon_depth_array(RDDMS_REST_API, dataspace_name, resqml_datatype, horizon_uuid, arrays_response, headers):
        arr_list = arrays_response.json() or []
        if not arr_list:
            raise SystemExit("No arrays found for the selected horizon")
        array_uuid_url = urllib.parse.quote(arr_list[0]['uid']['pathInResource'], safe="")
        r = api_request('GET', f"{RDDMS_REST_API}/dataspaces/{dataspace_name}/resources/{resqml_datatype}/{horizon_uuid}/arrays/{array_uuid_url}", params={'format':'json'}, headers=headers, tag='array-read')
        r.raise_for_status()
        return r

    arr = process_horizon_depth_array(RDDMS_REST_API, dataspace_name, resqml_datatype, horizon_uuid, arrays_response, headers=get_rddms_header())
    dims = arr.json()['data']['dimensions']
    z = np.array(arr.json()['data']['data'], dtype=np.float32)
    z2 = z.reshape(dims)
    stats = {"Min":[float(np.nanmin(z2))], "Max":[float(np.nanmax(z2))], "Mean":[float(np.nanmean(z2))], "Std Dev":[float(np.nanstd(z2))]}
    pretty_print_panda_frame(pd.DataFrame(stats), title="Depth Array Statistics", transpose=True, show_index=True, show_header=False)
else:
    print('Skipping depth array read: no horizons available.')


### 8. Plot 3D surface (Plotly)

In [ ]:

%pip -q install plotly
import plotly.graph_objects as go

if HAS_HORIZON and arrays_response is not None:
    # recompute X,Y using geometry lattice
    geom_meta, arrs = process_horizon_metadata(RDDMS_REST_API, dataspace_name, resqml_datatype, horizon_uuid, headers=get_rddms_header())
    sizes = geom_meta['Size']
    origin = geom_meta['Origin']
    u_vec, v_vec = np.array(geom_meta['Vector']).T
    i, j = np.meshgrid(np.arange(sizes[1]), np.arange(sizes[0]), indexing='ij')
    X = origin[0] + i * u_vec[0] + j * v_vec[0]
    Y = origin[1] + i * u_vec[1] + j * v_vec[1]

    # depth array
    import urllib.parse
    array_uuid_url = urllib.parse.quote(arrs.json()[0]['uid']['pathInResource'], safe="")
    arr = api_request('GET', f"{RDDMS_REST_API}/dataspaces/{dataspace_name}/resources/{resqml_datatype}/{horizon_uuid}/arrays/{array_uuid_url}", params={'format':'json'}, headers=get_rddms_header(), tag='array-read')
    arr.raise_for_status()
    dims = arr.json()['data']['dimensions']
    Z = np.reshape(np.array(arr.json()['data']['data'], dtype=np.float32), (int(dims[0]), int(dims[1])))

    z_min, z_max = float(np.nanmin(Z)), float(np.nanmax(Z))
    fig = go.Figure(data=[go.Surface(z=Z, x=X, y=Y, colorscale='Viridis', cmin=z_min, cmax=z_max)])
    fig.update_layout(title='3D Plot of Seismic Horizon', scene=dict(xaxis_title='X (m)', yaxis_title='Y (m)', zaxis_title='Depth (m)', zaxis=dict(range=[z_max, z_min])), width=800, height=700)
    fig.show()
else:
    print('Skipping plot: no horizons available.')


## 9. Read several horizons (2D grid) with FETPAPI/FESAPI

### 9.1 Setup & open ETP websocket (robust)

In [ ]:

%pip -q install fetpapi
import fesapi, fetpapi, uuid, time, urllib.parse, numpy as np

# 1) Create ETP session
initialization_params = fetpapi.InitializationParameters(str(uuid.uuid4()), RDDMS_ETP_API)

# Handshake header (lower-case key)
additionalHeaderField = fetpapi.MapStringString()
additionalHeaderField["data-partition-id"] = DATA_PARTITION_ID
initialization_params.setAdditionalHandshakeHeaderFields(additionalHeaderField)

client_session = fetpapi.createClientSession(initialization_params, f"Bearer {get_token()}")

# 2) Run the session in a background thread
from threading import Thread

def start_etp_server(sess):
    sess.run()

Thread(target=start_etp_server, args=(client_session,), daemon=True).start()

# 3) Wait for handshake to complete (up to 30 s)
deadline = time.time() + 30.0
while client_session.isEtpSessionClosed() and time.time() < deadline:
    time.sleep(0.25)

if client_session.isEtpSessionClosed():
    raise SystemExit("ETP session could not be established in 30 seconds. "
                     "Check WSS URL, token, data-partition-id header, and TLS.")

# Give handlers a tick to register
time.sleep(0.5)
print(f"Now connected to OSDU RDDMS at {RDDMS_ETP_API}")


### 9.2 Load dataspace resources into the Repository (no arrays yet)

In [ ]:

# EML dataspace URI built from the selected dataspace path (no regex)
dataspace_uri_path = dataspace_path  # e.g., "maap/drogon"

etp_context = fetpapi.ContextInfo()
etp_context.uri = f"eml:///dataspace('{dataspace_uri_path}')"
etp_context.depth = 1
etp_context.navigableEdges = fetpapi.RelationshipKind_Both
etp_context.includeSecondaryTargets = False
etp_context.includeSecondarySources = False

# Helper: retry getResources in case the very first request races the handshake

def get_resources_with_retry(sess, ctx, scope, *, retries=3, backoff=1.5):
    for attempt in range(1, retries+1):
        try:
            return sess.getResources(ctx, scope)
        except RuntimeError as e:
            if "Time out waiting for a response" in str(e) and attempt < retries:
                time.sleep(backoff * attempt)
                continue
            raise

all_resources = get_resources_with_retry(client_session, etp_context, fetpapi.ContextScopeKind__self)

if all_resources.empty():
    print(f"There is no resource in dataspace {dataspace_uri_path}")
else:
    print(f"There are {len(all_resources)} resources in dataspace {dataspace_uri_path}")

# Build ETP URIs for horizons from the REST list, if available
etp_uris = [x['uri'] for x in (hlist or [])]

# Get all data objects from the resources
uriMap = fetpapi.MapStringString()
for index, resource in enumerate(all_resources):
    uriMap[str(index)] = resource.uri
all_objects = client_session.getDataObjects(uriMap)

repo = fesapi.DataObjectRepository()
repo.setHdfProxyFactory(fetpapi.FesapiHdfProxyFactory(client_session))
for dataObject in all_objects.values():
    repo.addOrReplaceGsoapProxy(
        dataObject.data,
        fetpapi.getDataObjectType(dataObject.resource.uri),
        fetpapi.getDataspaceUri(dataObject.resource.uri)
    )
print("Dataspace loaded in memory.")


### 9.3 Read horizon Z arrays (stats)

In [ ]:

z_data = {}
for etp_uri in etp_uris:
    hz_uuid = urllib.parse.quote(etp_uri.split('(')[-1].replace(')',''))
    horizon = repo.getDataObjectByUuid(hz_uuid)
    if not horizon:
        print(f"Horizon not found in repo for UUID: {hz_uuid}")
        continue
    grid2dNodeCount = horizon.getXyzPointCountOfAllPatches()
    if isinstance(horizon, fesapi.Resqml2_Grid2dRepresentation):
        z_values = fesapi.DoubleArray(grid2dNodeCount)
        horizon.getZValues(z_values)
        arr = np.empty(grid2dNodeCount, dtype=np.float64)
        for i in range(grid2dNodeCount):
            arr[i] = z_values.getitem(i)
        z_data[horizon] = arr
        # Stats
        stats = {"Min":[float(np.nanmin(arr))], "Max":[float(np.nanmax(arr))], "Mean":[float(np.nanmean(arr))], "Std Dev":[float(np.nanstd(arr))]}
        pretty_print_panda_frame(pd.DataFrame(stats), title=f"Depth Array Statistics of {horizon.getTitle()}", transpose=True, show_index=True, show_header=False)
    else:
        print(f"UUID {hz_uuid} is not a Grid2d horizon.")


### 9.4 Read the first 5 faults

In [ ]:

fault_x_data, fault_y_data, fault_z_data = {}, {}, {}
all_faults_count = repo.getFaultPolylineSetRepresentationCount()
for fault_index in range(min(5, all_faults_count)):
    fault = repo.getFaultPolylineSetRepresentation(fault_index)
    fault_x_data[fault], fault_y_data[fault], fault_z_data[fault] = [], [], []

    node_count = fault.getXyzPointCountOfAllPatches()
    xyz_values = fesapi.DoubleArray(node_count * 3)
    fault.getXyzPointsOfAllPatches(xyz_values)

    polyline_count = fault.getPolylineCountOfAllPatches()
    node_count_per_polyline = fesapi.UInt32Array(polyline_count)
    fault.getNodeCountPerPolylineOfAllPatches(node_count_per_polyline.cast())

    node_index = 0
    for polyline_index in range(polyline_count):
        plc = node_count_per_polyline.getitem(polyline_index)
        fault_x_data[fault].append(np.empty(plc, dtype=np.float64))
        fault_y_data[fault].append(np.empty(plc, dtype=np.float64))
        fault_z_data[fault].append(np.empty(plc, dtype=np.float64))
        for poly_node in range(plc):
            fault_x_data[fault][polyline_index][poly_node] = xyz_values.getitem(node_index*3)
            fault_y_data[fault][polyline_index][poly_node] = xyz_values.getitem(node_index*3+1)
            fault_z_data[fault][polyline_index][poly_node] = xyz_values.getitem(node_index*3+2)
            node_index += 1
print("Faults loaded in memory.")


### 9.5 Close the ETP websocket session

In [ ]:

client_session.close()
print("ETP session is now closed")


### 9.6 Plot horizons & faults (Plotly)

In [ ]:

import plotly.graph_objects as go
surfaces = []

# Determine global z range
zmins, zmaxs = [], []
for horizon, arr in z_data.items():
    Z = arr.reshape(horizon.getNodeCountAlongJAxis(), horizon.getNodeCountAlongIAxis()).transpose()
    if np.nanmin(Z)==np.nanmax(Z)==0: continue
    zmins.append(np.nanmin(Z)); zmaxs.append(np.nanmax(Z))

global_z_min, global_z_max = (float(np.nanmin(zmins)), float(np.nanmax(zmaxs))) if (zmins and zmaxs) else (0.0, 1.0)

# Horizons
show_scale = True
for horizon, arr in z_data.items():
    i, j = np.meshgrid(np.arange(horizon.getNodeCountAlongIAxis()), np.arange(horizon.getNodeCountAlongJAxis()), indexing='ij')
    X = horizon.getXOrigin() + i * horizon.getXIOffset() + j * horizon.getXJOffset()
    Y = horizon.getYOrigin() + i * horizon.getYIOffset() + j * horizon.getYJOffset()
    Z = arr.reshape(horizon.getNodeCountAlongJAxis(), horizon.getNodeCountAlongIAxis()).transpose()
    surfaces.append(go.Surface(z=Z, x=X, y=Y, colorscale='Viridis', cmin=global_z_min, cmax=global_z_max, showscale=show_scale, colorbar=dict(title="Depth")))
    show_scale = False

# Faults (polyline traces)
import random
for fault, polyset in fault_z_data.items():
    rand_color = f'rgb({random.randint(0,255)},{random.randint(0,255)},{random.randint(0,255)})'
    for idx, z_arr in enumerate(polyset):
        surfaces.append(go.Scatter3d(z=z_arr, x=fault_x_data[fault][idx], y=fault_y_data[fault][idx], mode='lines', line=dict(color=rand_color, width=2), showlegend=False))

fig = go.Figure(data=surfaces)
fig.update_layout(title='3D Horizons + Faults (FESAPI)', scene=dict(xaxis_title='X (m)', yaxis_title='Y (m)', zaxis_title='Depth (m)', zaxis=dict(range=[global_z_max, global_z_min])), width=1200, height=700)
fig.show()


## 10. Using the OSDU ETP client Docker image

### 10.1 Pull the Docker image and set connection credentials

In [ ]:

# NOTE: Requires Docker CLI available in your environment

docker_cmd = "docker"
!{docker_cmd} pull --quiet community.opengroup.org:5555/osdu/platform/domain-data-mgmt-services/reservoir/open-etp-server/open-etp-sslclient-main
!{docker_cmd} tag community.opengroup.org:5555/osdu/platform/domain-data-mgmt-services/reservoir/open-etp-server/open-etp-sslclient-main open-etp-sslclient

etp_credentials = f"--server-url {RDDMS_ETP_API} --data-partition-id {DATA_PARTITION_ID} --auth bearer --jwt-token {get_token()}"
space_root_cmd = f"/bin/openETPServer space {etp_credentials}"
print('Docker image is ready to be used.')


### 10.2 List all dataspaces

In [ ]:

list_dataspace_cmd = f"{space_root_cmd} space --list"
!{docker_cmd} run --rm --entrypoint=sh open-etp-sslclient -c "{list_dataspace_cmd}"


### 10.3 (Optional) Import a sample RESQML into a dataspace

In [ ]:

# Optional sample import
import os, requests, json as _json

user_dataspace_name = f"rddms_lab/Test_Dataspace_{user['handle']}"
xdata_acl = {
  "legaltags": [f"{DEFAULT_LEGAL_TAG}"],
  "otherRelevantDataCountries": DEFAULT_OTHER_RELEVANT_DATA_COUNTRIES,
  "owners": DEFAULT_OWNERS,
  "viewers": DEFAULT_VIEWERS,
}

xdata_acl_json = _json.dumps(xdata_acl).replace('"', '\\"')
new_space_cmd = f"{space_root_cmd} space --new -s {user_dataspace_name} --xdata '{xdata_acl_json}'"
!{docker_cmd} run --rm --entrypoint=sh open-etp-sslclient -c "{new_space_cmd}"

repo = "https://community.opengroup.org/osdu/platform/domain-data-mgmt-services/reservoir/open-etp-server/-/raw/main/data/"
epc_file = "Volve_Demo_Faults_Depth.epc"; hdf5_file = "Volve_Demo_Faults_Depth.h5"
for name in (hdf5_file, epc_file):
    url = f"{repo}{name}"; 
    if not os.path.exists(name):
        r = requests.get(url, stream=True); r.raise_for_status()
        with open(name, 'wb') as f: f.write(r.content)
        print(f"Downloaded {name}")
    else:
        print(f"File exists: {name}")

import_resqml_cmd = f"{space_root_cmd} space -s {user_dataspace_name} --import-epc /data/{epc_file}"
!{docker_cmd} run --rm -v .:/data --entrypoint=sh open-etp-sslclient -c "{import_resqml_cmd}"

check_cmd = f"{space_root_cmd} space -s {user_dataspace_name} --stats"
!{docker_cmd} run --rm --entrypoint=sh open-etp-sslclient -c "{check_cmd}"
